# 04 - 端到端流程（冒烟 / 全量 F1 最大化 / 全量 sid + 断点续跑）

本 Notebook 是 **`docs/user_guide.md`** 中命令行流水线的交互版，支持 **marker 断点续跑**（日志在 `runs/pipeline-logs/`）。

**本 Notebook 侧重项目 KPI：验证集 F1 最大化**（`early_stop_metric: best_f1` + `threshold_search.objective: f1`）。  
召回率优先 / 精度优先全量实验见 **`05_full_recall_vs_precision.ipynb`**。

## 三套流程

| 流程 | marker 前缀 | 实验 YAML | 说明 |
|------|-------------|-----------|------|
| **冒烟** | `smoke` | `exp_baseline.yaml` | ~500 行索引；**关闭 DDP** |
| **全量 F1 time** | `full_f1` | `exp_full_f1_ddp.yaml` | 全量索引 + **time_stratified**；**F1 最大化** |
| **全量 F1 sid** | `full_f1_sid` | `exp_full_f1_sid.yaml` | **共用 processed/**；`resplit.sh` → 跨机泛化 |

**推荐顺序**：冒烟 → 全量 F1 time（含 preprocess）→ **全量 F1 sid**（仅 resplit + 训练，不重复扫 mcelog）。

## 产物与指标

- 训练/推理：`runs/exp_tianchi_full_f1_ddp/` 与 `runs/exp_tianchi_full_f1_sid/`
- **部署 KPI**：`metrics.json` 顶层 / `deployment`（验证集 **F1 最优阈值**）；**勿**用 `oracle_on_test` 作线上承诺
- 误差分析：`03_error_analysis.ipynb`（FN/FP × `empty_mcelog_window` × sid）

## 使用提示

- marker 前缀 **≠** `run_name`（例：`full_f1_sid` vs `exp_tianchi_full_f1_sid`）
- sid 单元要求实验 YAML 中 `processed_dir/preprocess_manifest.json` 存在（先跑全量 time 的 preprocess）
- 改索引/horizon：清空 `data/processed/*` 与各 `splits*` 并删相关 marker
- **源码更新后 import 报错**：先运行下方 setup 单元，或 **Kernel → Restart** 后从头执行

命令行：冒烟 **`bash scripts/autotune.sh`**；全量 F1 **`exp_full_f1_*.yaml`**；sid **`bash scripts/resplit.sh`**。

In [1]:
import os
from pathlib import Path

from dram2cnn.utils.notebook_bootstrap import notebook_setup
from dram2cnn.utils.pipeline_helpers import pipeline_log_dir, pipeline_marker_dir

PROJECT_ROOT = notebook_setup(cwd=Path.cwd().resolve())
LOG_DIR = pipeline_log_dir(PROJECT_ROOT)
MARKER_DIR = pipeline_marker_dir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOG_DIR:", LOG_DIR)
print("MARKER_DIR:", MARKER_DIR)


PROJECT_ROOT: /root/workspace
LOG_DIR: /root/workspace/runs/pipeline-logs
MARKER_DIR: /root/workspace/runs/pipeline-logs/markers


In [2]:
from dram2cnn.utils.pipeline_helpers import (
    clear_exp_markers,
    preflight_full_scale_index,
    preflight_split_labels,
    preflight_splits_vs_model,
    require_preprocess_manifest,
    run_step,
)
from dram2cnn.utils.deployment_kpi import (
    F1_KPI_PAIRS,
    collect_kpi_rows,
    merge_all_strategies_kpi,
    write_kpi_csv,
)

# 实验 YAML / marker 前缀（全 notebook 共用，避免后续单元 NameError）
SMOKE_EXP = "smoke"
FULL_EXP = "full_f1"
SID_EXP = "full_f1_sid"
SMOKE_CFG = "configs/experiments/exp_baseline.yaml"
FULL_CFG = "configs/experiments/exp_full_f1_ddp.yaml"
SID_CFG = "configs/experiments/exp_full_f1_sid.yaml"

print("pipeline helpers loaded (run_step, preflight, deployment_kpi, 实验常量)")

pipeline helpers loaded (run_step, preflight, deployment_kpi, 实验常量)


---
## 冒烟测试（Smoke）

目标：快速验证 **索引 → 预处理 → 训练 → 推理 → 评估** 全链路可用。

| 步骤 | 说明 |
|------|------|
| 1. 生成演示索引 | `features_5m_index_demo.csv`（~500 行） |
| 2. preprocess | `exp_baseline.yaml`，截断 mcelog/索引 |
| 3. train | `train.sh`（exp_baseline **关闭 DDP** 以加速） |
| 4. infer / evaluate | 推理与指标 |

命令行一键等价（无 marker）：**`bash scripts/autotune.sh`** = preprocess（含 validate）→ train → infer → evaluate（**不含** build_index/inspect_data；完整冒烟见本 Notebook 冒烟单元）。

In [3]:
# 【冒烟 1/2】重建演示索引（与 configs/data.yaml 默认 index_features_path 一致）
run_step(SMOKE_EXP, "build_index", "bash scripts/build_index.sh smoke")

[RUN ] smoke/build_index
      command: bash scripts/build_index.sh smoke
      log: /root/workspace/runs/pipeline-logs/smoke__build_index__20260616_220259.log
Building index mode=smoke horizon=300.0s interval=300.0s window=300.0s pos_interval=60.0s → /root/workspace/data/index_stubs/features_5m_index_demo.csv
/root/workspace/src/dram2cnn/data/index_builder.py:632: UserWarning: mcelog_max_rows=800000：仅扫描 mcelog 前若干行计算 time_epoch_offset，可能不是全文件全局最小值；冒烟/截断模式不应用于生产预处理。
  time_epoch_offset = compute_global_time_epoch_offset(
Done: rows=500 pos=43 neg=457 pos_rate=0.0860
[DONE] smoke/build_index



In [4]:
# 【冒烟 2/2】baseline 全流程（configs/experiments/exp_baseline.yaml）

run_step(SMOKE_EXP, "preprocess", f"bash scripts/preprocess.sh {SMOKE_CFG}")
# preprocess.sh 末尾已 validate；以下为可选重复诊断
# run_step(SMOKE_EXP, "validate", f"bash scripts/validate.sh {SMOKE_CFG}")
run_step(SMOKE_EXP, "inspect_data", f"bash scripts/inspect_data.sh {SMOKE_CFG}")

preflight_splits_vs_model(SMOKE_CFG, PROJECT_ROOT)
preflight_split_labels(SMOKE_CFG, PROJECT_ROOT)

run_step(SMOKE_EXP, "train", f"bash scripts/train.sh {SMOKE_CFG}")
run_step(SMOKE_EXP, "infer", f"bash scripts/infer.sh {SMOKE_CFG}")
run_step(SMOKE_EXP, "evaluate", f"bash scripts/evaluate.sh {SMOKE_CFG}")

print("冒烟完成 → runs/exp_tianchi_baseline/")
print("后续分析：02_feature_analysis.ipynb（X 分布）、03_error_analysis.ipynb（FP/FN，EXP 仍用 exp_baseline）")

[RUN ] smoke/preprocess
      command: bash scripts/preprocess.sh configs/experiments/exp_baseline.yaml
      log: /root/workspace/runs/pipeline-logs/smoke__preprocess__20260616_220305.log
Scanning mcelog for row/col max …
Grid representation='spatial_heatmap', (H,W)=(128,32), channels=31
Computing global time epoch offset (earliest raw timestamp → relative t=0) …
/root/workspace/src/dram2cnn/data/tianchi_build.py:662: UserWarning: mcelog_max_rows=800000：仅扫描 mcelog 前若干行计算 time_epoch_offset，可能不是全文件全局最小值；冒烟/截断模式不应用于生产预处理。
  time_epoch_offset = compute_global_time_epoch_offset(
  time_epoch_offset_absolute=-62135596798.000
Loading index CSV …
Loading inventory one-hot …
预处理并行度: worker 预算=67（CPU 上限=67，floor(112×0.6)，RAM 预算=31.2GiB（62GiB×0.5，目标≤50%），估算峰值≈6.3GiB；split/归一化估算≈3.1GiB；顺序读 mcelog 仍为单线程；Linux 上对排序/填网格 fork 并行）
Building mcelog time index（顺序流式读 CSV + 按 sid 聚合；排序/填网格阶段见上并行度）…
Sorting mcelog buckets by time (fork Pool, 65 tasks, ~67 workers budget) …
Filling grids with fork Pool (67 t

In [6]:
# 【可选】命令行 autotune 一键冒烟（不分 marker；与上一单元二选一即可）
# run_step(SMOKE_EXP, "autotune", "bash scripts/autotune.sh")

---
## 全量 F1 最大化（Full + DDP）

目标：在 **真实全量索引 + 完整 mcelog** 上训练，**早停与阈值搜索均最大化验证集 F1**（项目 KPI 口径）。

| 步骤 | 说明 |
|------|------|
| 1. 生成全量索引 | 输出 `data/raw/features_index_h300s_step300s_full.csv`（耗时长） |
| 2. **配置** | `exp_full_f1_ddp.yaml`；`drop_empty_positive_windows: false`（与历史 F1 基线一致） |
| 3. 清空旧产物 | 若曾跑工程向 preprocess：`rm -rf data/processed/* data/splits/*` 及 `full_f1__preprocess.done` |
| 4. preprocess → evaluate | **`train_ddp.sh`** 多卡；产物 `runs/exp_tianchi_full_f1_ddp/` |

> 终端等价：`bash scripts/preprocess.sh configs/experiments/exp_full_f1_ddp.yaml`

In [5]:
# 【全量 F1 1/3】扫描完整 mcelog 生成全量索引（耗时长，可跳过若 CSV 已存在且 marker 在）
run_step(FULL_EXP, "build_index", "bash scripts/build_index.sh full")

print(
    "\n下一步：\n"
    "  1) exp_full_f1_ddp.yaml → index_features_path=data/raw/features_index_h300s_step300s_full.csv\n"
    "  2) 若曾跑过工程向 preprocess：rm -rf data/processed/* data/splits/*\n"
    "  3) clear_exp_markers('full_f1')  # 或删 full_f1__preprocess.done\n"
    "  4) 运行下方「全量 F1 预检 + 流水线」单元"
)

[RUN ] full_f1/build_index
      command: bash scripts/build_index.sh full
      log: /root/workspace/runs/pipeline-logs/full_f1__build_index__20260616_220531.log
Building index mode=full horizon=300.0s interval=300.0s window=300.0s pos_interval=60.0s → /root/workspace/data/raw/features_index_h300s_step300s_full.csv
Done: rows=66320 pos=9948 neg=56372 pos_rate=0.1500
[DONE] full_f1/build_index


下一步：
  1) exp_full_f1_ddp.yaml → index_features_path=data/raw/features_index_h300s_step300s_full.csv
  2) 若曾跑过工程向 preprocess：rm -rf data/processed/* data/splits/*
  3) clear_exp_markers('full_f1')  # 或删 full_f1__preprocess.done
  4) 运行下方「全量 F1 预检 + 流水线」单元


In [6]:
# 【全量 F1 2/3】预检：index_features_path 不得仍指向演示 stub
_cfg = preflight_full_scale_index(FULL_CFG, root=PROJECT_ROOT, context="04_full_f1_preflight")
print(f"[preflight] 全量 index_features_path OK: {_cfg['data']['index_features_path']}")
print(f"[preflight] early_stop_metric={_cfg['train']['early_stop_metric']} threshold_objective={_cfg['train']['threshold_search']['objective']}")

[preflight] 全量 index_features_path OK: /root/workspace/data/raw/features_index_h300s_step300s_full.csv
[preflight] early_stop_metric=best_f1 threshold_objective=f1


In [7]:
# 【全量 F1 3/3】exp_full_f1_ddp：preprocess → DDP 训练 → infer → evaluate
run_step(FULL_EXP, "preprocess", f"bash scripts/preprocess.sh {FULL_CFG}")
run_step(FULL_EXP, "inspect_data", f"bash scripts/inspect_data.sh {FULL_CFG}")

preflight_splits_vs_model(FULL_CFG, PROJECT_ROOT)
preflight_split_labels(FULL_CFG, PROJECT_ROOT)

run_step(FULL_EXP, "train", f"bash scripts/train_ddp.sh {FULL_CFG}")
run_step(FULL_EXP, "infer", f"bash scripts/infer.sh {FULL_CFG}")
run_step(FULL_EXP, "evaluate", f"bash scripts/evaluate.sh {FULL_CFG}")

print("全量 F1 time 完成 → runs/exp_tianchi_full_f1_ddp/")
print("分析 03 时请将 EXP_YAML 改为 exp_full_f1_ddp.yaml")

[RUN ] full_f1/preprocess
      command: bash scripts/preprocess.sh configs/experiments/exp_full_f1_ddp.yaml
      log: /root/workspace/runs/pipeline-logs/full_f1__preprocess__20260616_221019.log
Scanning mcelog for row/col max …
Grid representation='spatial_heatmap', (H,W)=(128,32), channels=31
Computing global time epoch offset (earliest raw timestamp → relative t=0) …
  time_epoch_offset_absolute=-62135596800.000
Loading index CSV …
Loading inventory one-hot …
预处理并行度: worker 预算=1（CPU 上限=67，floor(112×0.6)，RAM 预算=31.2GiB（62GiB×0.5，目标≤50%），估算峰值≈38.0GiB，已由 67 压至 1，警告：即使最小 worker 估算峰值仍超预算（全量 X 体量过大，请增内存或减索引）；split/归一化估算≈7.3GiB；顺序读 mcelog 仍为单线程；Linux 上对排序/填网格 fork 并行）
Building mcelog time index（顺序流式读 CSV + 按 sid 聚合；排序/填网格阶段见上并行度）…
警告：35260/66320 条样本在 mcelog 窗口内无事件（sid 缺失或窗口为空），对应 X 行为全零。
已丢弃 30666/66320 条空 mcelog 窗口样本（负样本空窗；正样本空窗保留 4594 条）。
Preprocess done: N=35654, pos_rate=0.279015
Preprocess + split completed. [31, 128, 32]
Validate after preprocess: ok
[DONE] full_f1/preprocess

[RUN 

---
## 全量 F1 sid 泛化对比

目标：在**同一** `processed_dir` 上按 **sid** 切分，与 **time_stratified**（`full_f1`）对比 **deployment F1**。

| 步骤 | 说明 |
|------|------|
| 0. 前置 | 已完成 **全量 F1 time** 的 `preprocess` |
| 1. resplit → evaluate | 产物 `runs/exp_tianchi_full_f1_sid/` |

命令行等价：**`bash scripts/resplit.sh configs/experiments/exp_full_f1_sid.yaml`**

In [8]:
# 【F1 sid 1/2】resplit → train → infer → evaluate

_proc_man = require_preprocess_manifest(
    FULL_CFG,
    root=PROJECT_ROOT,
    hint_cmd=f"bash scripts/preprocess.sh {FULL_CFG}",
)
print("[preflight] processed OK:", _proc_man)

run_step(SID_EXP, "resplit", f"bash scripts/resplit.sh {SID_CFG}")
run_step(SID_EXP, "inspect_data", f"bash scripts/inspect_data.sh {SID_CFG}")

preflight_splits_vs_model(SID_CFG, PROJECT_ROOT)
preflight_split_labels(SID_CFG, PROJECT_ROOT)

run_step(SID_EXP, "train", f"bash scripts/train_ddp.sh {SID_CFG}")
run_step(SID_EXP, "infer", f"bash scripts/infer.sh {SID_CFG}")
run_step(SID_EXP, "evaluate", f"bash scripts/evaluate.sh {SID_CFG}")

print("全量 F1 sid 完成 → runs/exp_tianchi_full_f1_sid/")
print("分析：03_error_analysis.ipynb，EXP_YAML=exp_full_f1_sid.yaml")

[preflight] processed OK: /root/workspace/data/processed/preprocess_manifest.json
[RUN ] full_f1_sid/resplit
      command: bash scripts/resplit.sh configs/experiments/exp_full_f1_sid.yaml
      log: /root/workspace/runs/pipeline-logs/full_f1_sid__resplit__20260616_224706.log
Split-only: 复用 processed/（n_samples=35654），策略=sid
Preprocess + split completed. [31, 128, 32]
Validate after preprocess: ok
[DONE] full_f1_sid/resplit

[RUN ] full_f1_sid/inspect_data
      command: bash scripts/inspect_data.sh configs/experiments/exp_full_f1_sid.yaml
      log: /root/workspace/runs/pipeline-logs/full_f1_sid__inspect_data__20260616_225947.log
实验配置: configs/experiments/exp_full_f1_sid.yaml
processed_dir: /root/workspace/data/processed
=== 数据概览（processed，原始特征尺度）===
提示：训练/验证/测试实际使用 splits/X_*.npy（切分后按训练集 mean/std 归一化）。此处为预处理全量池。
X shape: (35654, 31, 128, 32)
y shape: (35654,)
样本总数: 35654
正样本: 9948 (0.279015)
负样本: 25706 (0.720985)
meta.csv: /root/workspace/data/processed/meta.csv

=== meta 前3行 ===
   

In [9]:
# 【F1 sid 2/2】time vs sid：部署 KPI 对比
rows = collect_kpi_rows(F1_KPI_PAIRS, project_root_path=PROJECT_ROOT)
if rows:
    import pandas as pd

    cmp_df = pd.DataFrame(rows)
    print(cmp_df.to_string(index=False))
    out_cmp = PROJECT_ROOT / "outputs" / "tables"
    write_kpi_csv(rows, out_cmp / "full_f1_time_vs_sid_deployment_kpi.csv")
    merged = merge_all_strategies_kpi(tables_dir=out_cmp)
    print("\n已写入 outputs/tables/full_f1_time_vs_sid_deployment_kpi.csv")
    if merged:
        print(f"已更新汇总表 {merged.name}")

strategy                run_name  threshold       f1  precision   recall  roc_auc                                                                           metrics_path
 F1/time exp_tianchi_full_f1_ddp       0.70 0.761290   0.818182 0.711796 0.897871 /root/workspace/runs/exp_tianchi_full_f1_ddp/infer_20260616_224655_159358/metrics.json
  F1/sid exp_tianchi_full_f1_sid       0.66 0.713844   0.733239 0.695448 0.882111 /root/workspace/runs/exp_tianchi_full_f1_sid/infer_20260616_231544_511294/metrics.json

已写入 outputs/tables/full_f1_time_vs_sid_deployment_kpi.csv
已更新汇总表 full_all_strategies_deployment_kpi.csv


In [10]:
# 查看最近日志与 marker
print("\n=== 最近日志（最多 15 条）===")
for p in sorted(LOG_DIR.glob("*.log"), key=lambda x: x.stat().st_mtime, reverse=True)[:15]:
    print(p.name)

print("\n=== 已完成 marker ===")
for p in sorted(MARKER_DIR.glob("*.done")):
    print(p.name, "=>", p.read_text(encoding="utf-8").strip())


=== 最近日志（最多 15 条）===
full_f1_sid__evaluate__20260616_231551.log
full_f1_sid__infer__20260616_231526.log
full_f1_sid__train__20260616_225950.log
full_f1_sid__inspect_data__20260616_225947.log
full_f1_sid__resplit__20260616_224706.log
full_f1__evaluate__20260616_224702.log
full_f1__infer__20260616_224637.log
full_f1__train__20260616_223225.log
full_f1__inspect_data__20260616_223222.log
full_f1__preprocess__20260616_221019.log
full_f1__build_index__20260616_220531.log
smoke__evaluate__20260616_220355.log
smoke__infer__20260616_220350.log
smoke__train__20260616_220323.log
smoke__inspect_data__20260616_220319.log

=== 已完成 marker ===
full_f1__build_index.done => 20260616_220531
full_f1__evaluate.done => 20260616_224702
full_f1__infer.done => 20260616_224637
full_f1__inspect_data.done => 20260616_223222
full_f1__preprocess.done => 20260616_221019
full_f1__train.done => 20260616_223225
full_f1_sid__evaluate.done => 20260616_231551
full_f1_sid__infer.done => 20260616_231526
full_f1_sid__inspec

In [11]:
# 重置 marker（可选 — 取消注释后再运行本格）
# clear_exp_markers("smoke")
# clear_exp_markers("full_f1")
# clear_exp_markers("full_f1_sid")
print("未删除 marker。需重跑某流程时取消上方 clear_exp_markers(...) 注释。")

未删除 marker。需重跑某流程时取消上方 clear_exp_markers(...) 注释。
